In [12]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl
import math
from typing import Dict, List, Optional

sys.path.append('../')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
df_train = pl.read_parquet('../../data/clustered/train_cluster_features.parquet')
df_val = pl.read_parquet('../../data/clustered/val_cluster_features.parquet')
df_test = pl.read_parquet('../../data/clustered/test_cluster_features.parquet')


print(f"Train shape: {df_train.shape}")
print(f"Val shape: {df_val.shape}")
print(f"Test shape: {df_test.shape}")


Train shape: (4892637, 476)
Val shape: (808077, 476)
Test shape: (1906221, 476)


In [16]:
df_train.head()

cluster_id,ts_start,dep_internal_count,dep_internal_male,dep_internal_female,dep_internal_other,dep_internal_young,dep_internal_adult,dep_internal_senior,dep_internal_age_available,dep_internal_model_iconic,dep_internal_duration_mean,dep_internal_duration_std,dep_internal_temperature_2m_mean,dep_internal_relative_humidity_2m_mean,dep_internal_dew_point_2m_mean,dep_internal_apparent_temperature_mean,dep_internal_precipitation_mean,dep_internal_weather_code_mean,dep_internal_pressure_msl_mean,dep_internal_cloud_cover_mean,dep_internal_cloud_cover_low_mean,dep_internal_cloud_cover_mid_mean,dep_internal_cloud_cover_high_mean,dep_internal_vapour_pressure_deficit_mean,dep_internal_wind_speed_10m_mean,dep_internal_wind_direction_10m_mean,dep_internal_wind_gusts_10m_mean,dep_internal_soil_temperature_0_to_7cm_mean,dep_internal_soil_moisture_0_to_7cm_mean,dep_internal_sunshine_duration_mean,dep_internal_direct_radiation_mean,dep_internal_temp_feel_diff_mean,dep_internal_weather_comfort_mean,dep_internal_is_day_count,dep_internal_temp_comfortable_count,dep_internal_temp_very_hot_count,…,dep_internal_young_same_hour_yesterday,dep_internal_young_same_hour_last_week,dep_internal_adult_same_hour_yesterday,dep_internal_adult_same_hour_last_week,dep_internal_senior_same_hour_yesterday,dep_internal_senior_same_hour_last_week,dep_internal_model_iconic_same_hour_yesterday,dep_internal_model_iconic_same_hour_last_week,dep_internal_duration_mean_same_hour_yesterday,dep_internal_duration_mean_same_hour_last_week,arr_internal_count_same_hour_yesterday,arr_internal_count_same_hour_last_week,arr_external_count_same_hour_yesterday,arr_external_count_same_hour_last_week,arr_internal_male_same_hour_yesterday,arr_internal_male_same_hour_last_week,arr_internal_female_same_hour_yesterday,arr_internal_female_same_hour_last_week,arr_internal_young_same_hour_yesterday,arr_internal_young_same_hour_last_week,arr_internal_adult_same_hour_yesterday,arr_internal_adult_same_hour_last_week,arr_internal_senior_same_hour_yesterday,arr_internal_senior_same_hour_last_week,arr_internal_model_iconic_same_hour_yesterday,arr_internal_model_iconic_same_hour_last_week,has_dep_internal_same_hour_yesterday,has_dep_internal_same_hour_last_week,has_dep_external_same_hour_yesterday,has_dep_external_same_hour_last_week,has_arr_internal_same_hour_yesterday,has_arr_internal_same_hour_last_week,has_arr_external_same_hour_yesterday,has_arr_external_same_hour_last_week,has_short_term_lags,has_daily_lags,has_weekly_lags
i32,datetime[μs],u32,u32,u32,u32,i32,i32,i32,i32,i32,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,u32,u32,u32,u32,u32,u32,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
0,2020-01-01 00:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,2020-01-01 00:30:00,1,1,0,0,0,0,0,0,1,3599.0,0.0,21.311001,87.557816,19.161001,22.377872,0.0,1.0,1010.700012,39.0,12.0,19.0,23.0,0.315351,16.363178,140.355865,27.719999,23.111,0.468,0.0,0.0,1.066872,1.0,0,1,0,…,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,2020-01-01 01:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
0,2020-01-01 01:30:00,2,2,0,0,0,0,0,0,2,1710.5,26.162951,21.311001,87.557816,19.161001,22.377872,0.0,1.0,1010.700012,39.0,12.0,19.0,23.0,0.315351,16.363178,140.355865,27.719999,23.111,0.468,0.0,0.0,1.066872,1.0,0,2,0,…,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,2020-01-01 02:00:00,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,…,0,0,0,0,0,0,0,0,0.0,0.0,0,

In [17]:
df_train.columns

['cluster_id',
 'ts_start',
 'dep_internal_count',
 'dep_internal_male',
 'dep_internal_female',
 'dep_internal_other',
 'dep_internal_young',
 'dep_internal_adult',
 'dep_internal_senior',
 'dep_internal_age_available',
 'dep_internal_model_iconic',
 'dep_internal_duration_mean',
 'dep_internal_duration_std',
 'dep_internal_temperature_2m_mean',
 'dep_internal_relative_humidity_2m_mean',
 'dep_internal_dew_point_2m_mean',
 'dep_internal_apparent_temperature_mean',
 'dep_internal_precipitation_mean',
 'dep_internal_weather_code_mean',
 'dep_internal_pressure_msl_mean',
 'dep_internal_cloud_cover_mean',
 'dep_internal_cloud_cover_low_mean',
 'dep_internal_cloud_cover_mid_mean',
 'dep_internal_cloud_cover_high_mean',
 'dep_internal_vapour_pressure_deficit_mean',
 'dep_internal_wind_speed_10m_mean',
 'dep_internal_wind_direction_10m_mean',
 'dep_internal_wind_gusts_10m_mean',
 'dep_internal_soil_temperature_0_to_7cm_mean',
 'dep_internal_soil_moisture_0_to_7cm_mean',
 'dep_internal_sunshi